In [ ]:
import os
import pickle
import prince
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.font_manager import FontProperties

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
version3_path = os.path.join(parent_dir, "Version3")

os.chdir(version3_path)

from utils.utils_v3 import *
from utils.plots import *
from utils.preprocess import preprocess, process_other

try:
    myfont = FontProperties(fname=r"/System/Library/Fonts/PingFang.ttc")
    sns.set(style="whitegrid", font=myfont.get_name())
except Exception as e:
    print(e)

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

dataA2 = pd.read_csv("./Data/A2.csv", low_memory=False)
dataA1 = pd.read_csv("./Data/A1.csv")

In [ ]:
select_lst = [
    # 月份是為了篩選每個月2萬筆
    '發生月份',

    '天候名稱', '光線名稱', 
    '道路類別-第1當事者-名稱', '速限-第1當事者', 
    '路面狀況-路面鋪裝名稱', '路面狀況-路面狀態名稱', '路面狀況-路面缺陷名稱',
    '道路障礙-障礙物名稱', '道路障礙-視距品質名稱', '道路障礙-視距名稱',
    '號誌-號誌種類名稱', '號誌-號誌動作名稱',
    '車道劃分設施-分道設施-快車道或一般車道間名稱', '車道劃分設施-分道設施-快慢車道間名稱', '車道劃分設施-分道設施-路面邊線名稱',
    '當事者屬-性-別名稱', '當事者事故發生時年齡',
    '保護裝備名稱', '行動電話或電腦或其他相類功能裝置名稱',
    '肇事逃逸類別名稱-是否肇逃',
    '死亡受傷人數',

    # 大類別
    '道路型態大類別名稱', '事故位置大類別名稱',
    '車道劃分設施-分向設施大類別名稱',
    '事故類型及型態大類別名稱', '當事者區分-類別-大類別名稱-車種', '當事者行動狀態大類別名稱',
    '車輛撞擊部位大類別名稱-最初', '車輛撞擊部位大類別名稱-其他',

    # 兩個欄位只有兩個觀察值不同
    '肇因研判大類別名稱-主要',
    # '肇因研判大類別名稱-個別',
    
    # 子類別
    '道路型態子類別名稱', '事故位置子類別名稱', '事故類型及型態子類別名稱', '肇因研判子類別名稱-主要',
    '當事者區分-類別-子類別名稱-車種', '當事者行動狀態子類別名稱', '車輛撞擊部位子類別名稱-最初',
    '車輛撞擊部位子類別名稱-其他', '肇因研判子類別名稱-個別',
]

In [ ]:
full_dataA1 = preprocess(dataA1, target='全部', lst=select_lst)
full_dataA2 = preprocess(dataA2, target='全部', lst=select_lst)
mapper_numpy, rbind_data, dummy_data, death, injuried, date_info = process_other(
    full_dataA1, full_dataA2, downsample=False, en=False, return_time=True)


In [ ]:
import os, time, pickle
import numpy as np
import pandas as pd
import importlib.util

# cwd 目前在 Version3；用檔案路徑載入 Models/utils/models.py（Version3/utils 沒有 models）
_models_path = os.path.join(parent_dir, "Models", "utils", "models.py")
_spec = importlib.util.spec_from_file_location("models_new", _models_path)
models_new = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(models_new)
run_dr_baseline = models_new.run_dr_baseline


In [ ]:
# 依發生時間排序（時間切分前提）；date_info 與 rbind_data/dummy_data/death 同 index
order = date_info.sort_values(['發生日期', '發生時間']).index
dummy_sorted = dummy_data.loc[order].reset_index(drop=True).astype(float)   # PCA/UMAP 用（數值 one-hot）
rbind_sorted = rbind_data.loc[order].reset_index(drop=True)                 # MCA 用（原始類別）
y_binary = np.where(np.asarray(death.loc[order]) > 0, 1, 0)

dr_inputs = {'pca': dummy_sorted, 'umap': dummy_sorted, 'mca': rbind_sorted}
name_map = {'pca': 'pca', 'umap': 'umap', 'mca': 'mca_only'}   # 檔名沿用（mca 存成 mca_only）
print('sorted:', dummy_sorted.shape, 'y1 ratio=', y_binary.mean().round(4))


In [ ]:
SEED = 42
TRAIN_UNDER_RATIO = 0.1   # 訓練集下採樣 少數/多數=0.1 → 10:1（與 OriginModel 一致）
for algo in ['logistic', 'svc', 'xgboost']:
    for dr in ['pca', 'umap', 'mca']:
        print(f'{dr} {algo} start')
        t = time.time()
        y_out, scores_out, idx_out = run_dr_baseline(dr_inputs[dr], y_binary,
                                                     dr=dr, algo=algo, random_state=SEED,
                                                     train_under_ratio=TRAIN_UNDER_RATIO)
        elapsed = time.time() - t
        save_dir = f"../CompareOther/{algo}"
        os.makedirs(save_dir, exist_ok=True)
        with open(f"{save_dir}/{name_map[dr]}_{algo}.pkl", "wb") as f:
            pickle.dump({'y': y_out, 'decision_scores': scores_out,
                         'indices': idx_out, 'elapsed_time': elapsed}, f)
        print(f'{dr} {algo} done in {elapsed:.1f}s')


## 5.4 統一評估（同一時間切點 + 同一批測試列）

這一段**直接在本 notebook 產生 apples-to-apples 的 5.4 表**，不必另外開 notebook。
所有方法（Mapper / one-hot / PCA / UMAP / MCA）在同一個全域時間切點下切分、對**完全相同的一批 test 列**評估，`n_test` 一致。

注意：route A 必須重新訓練（無法用上面 cell 5 存到 CompareOther 的舊 pkl 拼出來，因為那些是各自 70/30、test 列不同）。這裡會在完整資料上重訓（有 10:1 下採樣加速），請預留時間。

In [ ]:
# 以檔案路徑載入 unified_eval（cwd 目前在 Version3；模組內部會自動找到 Models/utils 的兄弟模組）
import importlib.util
_ue_path = os.path.join(parent_dir, "Models", "utils", "unified_eval.py")
_ue_spec = importlib.util.spec_from_file_location("unified_eval", _ue_path)
ue = importlib.util.module_from_spec(_ue_spec); _ue_spec.loader.exec_module(ue)

TRAIN_UNDER_RATIO = 0.1
THRESHOLD = 'youden'
METHODS = ('mapper', 'onehot', 'pca', 'umap', 'mca')   # 想省時可先拿掉 'umap','mca'

for algo in ["xgboost", "logistic", "svc"]:
    print(f"\n===== 5.4 (unified) algo = {algo} =====")
    tbl = ue.table_5_4_unified(algo, data_root="./Data", train_under_ratio=TRAIN_UNDER_RATIO,
                               threshold=THRESHOLD, methods=METHODS)
    display(tbl)